In [1]:
# Patch Admin HTML asset paths and JS logic issues
import pathlib

root = pathlib.Path(r'e:\Cheesboy web project')
admin_html_files = sorted(root.glob('Admin/*.html'))
root_links = {
    'index.html': '../index.html',
    'women.html': '../women.html',
    'men.html': '../men.html',
    'kids.html': '../kids.html',
    'beauty.html': '../beauty.html',
    'sales.html': '../sales.html',
    'account.html': '../account.html',
    'wishlist.html': '../wishlist.html',
    'cart.html': '../cart.html',
    'seller-login.html': '../seller-login.html',
    'third-party-login.html': '../third-party-login.html'
}

for path in admin_html_files:
    text = path.read_text(encoding='utf-8')
    updated = text
    updated = updated.replace('href="css/styles.css"', 'href="../css/styles.css"')
    updated = updated.replace('src="js/script.js"', 'src="../js/script.js"')
    updated = updated.replace('src="js/', 'src="../js/')
    for old, new in root_links.items():
        updated = updated.replace(f'href="{old}"', f'href="{new}"')
    if updated != text:
        path.write_text(updated, encoding='utf-8')
        print(f'Patched {path.name}')

# Patch js/script.js admin nav logic
script_path = root / 'js' / 'script.js'
script_text = script_path.read_text(encoding='utf-8')
script_updated = script_text.replace(
    'function updateAdminNavVisibility() {\n    const adminLinks = document.querySelectorAll(\'nav a[href$="admin-login.html"], nav a[href$="admin.html"]\');\n    if (!adminLinks.length) return;\n\n    const isAdminLoggedIn = sessionStorage.getItem(\'adminLoggedIn\') === \'true\';\n    const isLoginPage = window.location.pathname.toLowerCase().endsWith(\'admin-login.html\');\n\n    adminLinks.forEach(link => {\n        if (isLoginPage) {\n            link.style.display = \'inline-block\';\n            link.textContent = \'Admin\';\n            link.href = \'admin-login.html\';\n        } else {\n            link.style.display = \'none\';\n        }\n    });\n}\n',
    'function updateAdminNavVisibility() {\n    const adminLoginLinks = document.querySelectorAll(\'nav a[href$="admin-login.html"]\');\n    const adminDashboardLinks = document.querySelectorAll(\'nav a[href$="admin.html"]\');\n    const isAdminLoggedIn = sessionStorage.getItem(\'adminLoggedIn\') === \'true\';\n\n    adminLoginLinks.forEach(link => {\n        if (isAdminLoggedIn) {\n            link.href = \'admin.html\';\n        } else {\n            link.href = \'admin-login.html\';\n        }\n        link.style.display = \'inline-block\';\n    });\n\n    adminDashboardLinks.forEach(link => {\n        link.style.display = isAdminLoggedIn ? \'inline-block\' : \'none\';\n    });\n}\n'
)
if script_text != script_updated:
    script_path.write_text(script_updated, encoding='utf-8')
    print('Updated js/script.js')

# Patch js/admin.js and admin-login.js as before
admin_js = root / 'js' / 'admin.js'
admin_text = admin_js.read_text(encoding='utf-8')
admin_updated = admin_text.replace(
    'function showTab(tabName) {\n    // Hide all tab contents\n    const tabs = document.querySelectorAll(\'.tab-content\');\n    tabs.forEach(tab => tab.classList.remove(\'active\'));\n\n    // Remove active class from buttons\n    const buttons = document.querySelectorAll(\'.tab-button\');\n    buttons.forEach(button => button.classList.remove(\'active\'));\n\n    // Show selected tab\n    document.getElementById(tabName).classList.add(\'active\');\n\n    // Add active class to clicked button\n    event.target.classList.add(\'active\');\n}\n',
    'function showTab(tabName, evt) {\n    // Hide all tab contents\n    const tabs = document.querySelectorAll(\'.tab-content\');\n    tabs.forEach(tab => tab.classList.remove(\'active\'));\n\n    // Remove active class from buttons\n    const buttons = document.querySelectorAll(\'.tab-button\');\n    buttons.forEach(button => button.classList.remove(\'active\'));\n\n    // Show selected tab\n    const targetTab = document.getElementById(tabName);\n    if (targetTab) {\n        targetTab.classList.add(\'active\');\n    }\n\n    const activeButton = Array.from(buttons).find(btn => btn.dataset.tab === tabName);\n    if (activeButton) {\n        activeButton.classList.add(\'active\');\n    }\n}\n'
)
admin_updated = admin_updated.replace(
    "document.getElementById('upload-form').addEventListener('submit', function(e) {\n    e.preventDefault();\n    const fileInput = document.getElementById('video-upload');\n    const file = fileInput.files[0];\n    if (file) {\n        const url = URL.createObjectURL(file);\n        videos.push({ name: file.name, url: url });\n        localStorage.setItem('advertVideos', JSON.stringify(videos));\n        renderVideoList();\n        updateAdvertVideo();\n        fileInput.value = '';\n    }\n});\n",
    "const uploadForm = document.getElementById('upload-form');\nif (uploadForm) {\n    uploadForm.addEventListener('submit', function(e) {\n        e.preventDefault();\n        const fileInput = document.getElementById('video-upload');\n        const file = fileInput?.files?.[0];\n        if (file) {\n            const url = URL.createObjectURL(file);\n            videos.push({ name: file.name, url: url });\n            localStorage.setItem('advertVideos', JSON.stringify(videos));\n            renderVideoList();\n            updateAdvertVideo();\n            if (fileInput) fileInput.value = '';\n        }\n    });\n}\n"
)
admin_updated = admin_updated.replace(
    "function renderVideoList() {\n    const videoList = document.getElementById('video-list');\n    videoList.innerHTML = '';\n    videos.forEach((video, index) => {\n        const videoItem = document.createElement('div');\n        videoItem.className = 'video-item';\n        videoItem.innerHTML = `\n            <span>${video.name}</span>\n            <button onclick=\"removeVideo(${index})\">Remove</button>\n        `;\n        videoList.appendChild(videoItem);\n    });\n}\n",
    "function renderVideoList() {\n    const videoList = document.getElementById('video-list');\n    if (!videoList) return;\n    videoList.innerHTML = '';\n    videos.forEach((video, index) => {\n        const videoItem = document.createElement('div');\n        videoItem.className = 'video-item';\n        videoItem.innerHTML = `\n            <span>${video.name}</span>\n            <button onclick=\"removeVideo(${index})\">Remove</button>\n        `;\n        videoList.appendChild(videoItem);\n    });\n}\n"
)
admin_updated = admin_updated.replace(
    "function updateAdvertVideo() {\n    const videoElement = document.getElementById('advert-video');\n    if (videos.length > 0) {\n        // Cycle through videos every 10 seconds\n        let currentIndex = 0;\n        videoElement.src = videos[currentIndex].url;\n        setInterval(() => {\n            currentIndex = (currentIndex + 1) % videos.length;\n            videoElement.src = videos[currentIndex].url;\n        }, 10000); // 10 seconds\n    }\n}\n",
    "function updateAdvertVideo() {\n    const videoElement = document.getElementById('advert-video');\n    if (!videoElement || videos.length === 0) return;\n\n    let currentIndex = 0;\n    videoElement.src = videos[currentIndex].url;\n    setInterval(() => {\n        currentIndex = (currentIndex + 1) % videos.length;\n        videoElement.src = videos[currentIndex].url;\n    }, 10000); // 10 seconds\n}\n"
)
if admin_updated != admin_text:
    admin_js.write_text(admin_updated, encoding='utf-8')
    print('Updated js/admin.js')

admin_login = root / 'js' / 'admin-login.js'
admin_login_text = admin_login.read_text(encoding='utf-8')
admin_login_updated = admin_login_text.replace(
    "function showTab(tabName) {\n    const tabs = document.querySelectorAll('.tab-content');\n    tabs.forEach(tab => tab.classList.remove('active'));\n\n    const buttons = document.querySelectorAll('.tab-button');\n    buttons.forEach(button => button.classList.remove('active'));\n\n    document.getElementById(tabName).classList.add('active');\n    if (event && event.target) {\n        event.target.classList.add('active');\n    }\n}\n",
    "function showTab(tabName, evt) {\n    const tabs = document.querySelectorAll('.tab-content');\n    tabs.forEach(tab => tab.classList.remove('active'));\n\n    const buttons = document.querySelectorAll('.tab-button');\n    buttons.forEach(button => button.classList.remove('active'));\n\n    const targetTab = document.getElementById(tabName);\n    if (targetTab) targetTab.classList.add('active');\n\n    const activeButton = Array.from(buttons).find(btn => btn.dataset.tab === tabName);\n    if (activeButton) activeButton.classList.add('active');\n}\n"
)
if admin_login_text != admin_login_updated:
    admin_login.write_text(admin_login_updated, encoding='utf-8')
    print('Updated js/admin-login.js')

FileNotFoundError: [Errno 2] No such file or directory: 'e:\\Cheesboy web project\\js\\script.js'

In [ ]:
import pathlib

root = pathlib.Path(r'e:\Cheesboy web project')
html_files = list(root.glob('*.html')) + list(root.glob('Admin/*.html'))

for page in ['about.html', 'contact.html', 'privacy-policy.html']:
    p = root / page
    if not p.exists():
        p.write_text(f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{page.replace('.html','').replace('-',' ').title()} - Cheeseboy Fashion</title>
    <link rel="stylesheet" href="css/styles.css">
</head>
<body>
    <header>
        <div class="container">
            <div class="logo"><a href="index.html"><h1>Cheeseboy</h1></a></div>
            <div class="search-bar"><input type="text" placeholder="Search for products..."><button>Search</button></div>
            <div class="header-icons"><a href="account.html" class="icon">Account</a><a href="wishlist.html" class="icon">Wishlist</a><a href="#" class="icon">Cart</a></div>
        </div>
    </header>
    <nav><div class="container"><ul><li><a href="index.html">Home</a></li><li><a href="women.html">Women</a></li><li><a href="men.html">Men</a></li><li><a href="kids.html">Kids</a></li><li><a href="beauty.html">Beauty</a></li><li><a href="sales.html">Sale</a></li><li><a href="admin-login.html">Admin</a></li></ul></div></nav>
    <section class="account"><div class="container"><h2>{page.replace('.html','').replace('-',' ').title()}</h2><div class="account-form"><p>Welcome to our {page.replace('.html','').replace('-', ' ')} page.</p><p>Cheeseboy Fashion is committed to quality, service and secure shopping.</p></div></div></section>
    <footer><div class="container"><p>&copy; 2026 Cheeseboy Fashion powered by WAGWAN digitals</p><ul><li><a href="about.html">About Us</a></li><li><a href="admin-login.html">Admin</a></li><li><a href="contact.html">Contact</a></li><li><a href="privacy-policy.html">Privacy Policy</a></li></ul></div></footer>
</body>
</html>''', encoding='utf-8')

for page in html_files:
    text = page.read_text(encoding='utf-8')
    prefix = '../' if page.parent.name == 'Admin' else ''
    footer_links = f'''            <li><a href="{prefix}about.html">About Us</a></li>
                <li><a href="{prefix}admin-login.html">Admin</a></li>
                <li><a href="{prefix}contact.html">Contact</a></li>
                <li><a href="{prefix}privacy-policy.html">Privacy Policy</a></li>'''
    new_text = text
    if '<li><a href="#">About Us</a></li>' in new_text:
        new_text = new_text.replace('<li><a href="#">About Us</a></li>', footer_links)
        new_text = new_text.replace('<li><a href="#">Contact</a></li>', '')
        new_text = new_text.replace('<li><a href="#">Privacy Policy</a></li>', '')
    if new_text != text:
        page.write_text(new_text, encoding='utf-8')
